# Lab 4: Model-Based Deep RL

## TDDE78 — Deep Reinforcement Learning
**Linköping University, Spring 2026**

---

This lab covers two model-based RL approaches:

- **Dyna-Q**: augments DQN with a learned neural dynamics model for simulated planning
- **MCTS (UCT)**: uses tree search with environment simulations to select actions

Both methods are evaluated on **CartPole-v1** (4D state, 2 discrete actions).

> **Note:** The DQN components (Q-network, replay buffer, ε-greedy, Q-update) are already implemented and provided — these are the same as Lab 1. Your task is to implement the **WorldModel** and **MCTSPlanner**.

In [ ]:
import os
import copy
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from networks import QNetwork, WorldModel
from utils import ReplayBuffer, plot_dyna_results, plot_comparison, smooth

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

_here = globals().get('__vsc_ipynb_file__', os.path.abspath(''))
_nb_dir = os.path.dirname(_here) if os.path.isfile(_here) else _here
EXPERIMENTS_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'experiments'))
print(f'Experiments directory: {EXPERIMENTS_DIR}')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(42)
print('Setup complete!')

---

# Part A — Implementation

---

## DynaAgent (provided)

> **DQN is reused from Lab 1** — the Q-network, target network, replay buffer, and ε-greedy policy are already implemented in `DynaAgent` below. You do **not** re-implement DQN here.

The new component is the **WorldModel** (in `networks.py`): a neural network trained with supervised learning to predict (next state, reward, done) from (state, action).

Once trained, the WorldModel enables **Dyna-style planning**: generate `k` synthetic transitions per real step and use them as additional Q-update targets — without extra environment interaction.

In [ ]:
class DynaAgent:
    """
    Dyna-Q agent: DQN (provided, Lab 1) + WorldModel + simulated planning.

    The DQN components are fully provided. Your tasks are:
      1. Implement WorldModel in networks.py
      2. Implement _model_update() below
      3. Implement the planning loop in update()

    Args:
        state_dim       (int):   Observation space dimension.
        action_dim      (int):   Number of discrete actions.
        lr              (float): Learning rate for Q-network and world model.
        gamma           (float): Discount factor.
        buffer_capacity (int):   Replay buffer size.
        batch_size      (int):   Mini-batch size.
        planning_steps  (int):   Simulated Q-updates per real step (0 = pure DQN).
        target_update   (int):   Hard target-network update frequency (steps).
        eps_start       (float): Initial ε for ε-greedy exploration.
        eps_end         (float): Final ε.
        eps_decay       (float): Multiplicative ε decay per episode.
    """

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 buffer_capacity=50_000, batch_size=64, planning_steps=10,
                 target_update=200, eps_start=1.0, eps_end=0.01,
                 eps_decay=0.995, seed=42):
        self.action_dim     = action_dim
        self.gamma          = gamma
        self.batch_size     = batch_size
        self.planning_steps = planning_steps
        self.target_update  = target_update
        self.epsilon        = eps_start
        self.eps_end        = eps_end
        self.eps_decay      = eps_decay
        self._step          = 0

        # ── DQN (reused from Lab 1) ──────────────────────────────────
        self.q_net      = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.buffer     = ReplayBuffer(buffer_capacity, state_dim)
        self.q_opt      = torch.optim.Adam(self.q_net.parameters(), lr=lr)

        # ── WorldModel (new — implement in networks.py) ──────────────
        self.world_model = WorldModel(state_dim, action_dim).to(device)
        self.model_opt   = torch.optim.Adam(self.world_model.parameters(), lr=lr)

    # ── Provided: ε-greedy action selection ──────────────────────────
    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_dim)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            return self.q_net(state_t).argmax().item()

    # ── Provided: DQN Q-update ────────────────────────────────────────
    def _q_update(self, states, actions, rewards, next_states, dones):
        with torch.no_grad():
            next_q  = self.target_net(next_states).max(dim=1).values
            targets = rewards + self.gamma * (1.0 - dones) * next_q
        q_vals = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        loss   = F.mse_loss(q_vals, targets)
        self.q_opt.zero_grad()
        loss.backward()
        self.q_opt.step()
        self._step += 1
        if self._step % self.target_update == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        return loss.item()

    # ── TODO: WorldModel supervised update ───────────────────────────
    def _model_update(self, states, actions, rewards, next_states, dones):
        """
        Train the WorldModel on one batch of real transitions.

        Steps:
        1. Convert actions to one-hot:  F.one_hot(actions, self.action_dim).float()
        2. Forward: pred_ns, pred_r, pred_done_logit = self.world_model(states, a_oh)
        3. Losses:
             s_loss = F.mse_loss(pred_ns, next_states)
             r_loss = F.mse_loss(pred_r, rewards)
             d_loss = F.binary_cross_entropy_with_logits(pred_done_logit, dones)
             loss   = s_loss + r_loss + 0.1 * d_loss
        4. Backprop through self.model_opt
        5. Return loss.item()
        """
        raise NotImplementedError('Implement _model_update()')

    # ── TODO: Full Dyna update (real Q-step + model train + planning) ─
    def update(self):
        """
        Full Dyna update:
          1. Sample a batch from the real buffer
          2. Q-update on the real batch:    self._q_update(...)
          3. Model update on the real batch: self._model_update(...)
          4. Planning loop (repeat planning_steps times):
               a. Sample states/actions from the real buffer
               b. Generate synthetic transitions with world_model (no_grad)
               c. Convert done_logit to done flag: sigmoid > 0.5
               d. Q-update on synthetic transitions: self._q_update(...)
          5. Return (q_loss, model_loss)

        Return (0.0, 0.0) if buffer has fewer than batch_size transitions.
        """
        raise NotImplementedError('Implement update()')

print('DynaAgent defined')

In [ ]:
def train_dyna(env_name='CartPole-v1', total_timesteps=100_000,
               planning_steps=10, seed=42, solve_threshold=475.0,
               log_interval=20, **agent_kwargs):
    """
    Train a DynaAgent on a Gymnasium environment.

    With planning_steps=0 this is equivalent to a standard DQN (Lab 1 baseline).

    Args:
        env_name        (str):   Gymnasium environment id.
        total_timesteps (int):   Maximum environment steps.
        planning_steps  (int):   Simulated updates per real step.
        seed            (int):   Random seed.
        solve_threshold (float): Early-stop when avg reward (last 100 ep) >= this.
        log_interval    (int):   Print every N episodes.
        **agent_kwargs:          Passed to DynaAgent.

    Returns:
        dict with keys: episode_rewards, episode_timesteps, q_losses, model_losses, agent
    """
    set_seed(seed)
    env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = DynaAgent(state_dim, action_dim,
                      planning_steps=planning_steps, seed=seed, **agent_kwargs)

    episode_rewards   = []
    episode_timesteps = []
    q_losses          = []
    model_losses      = []

    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0

    for step in range(total_timesteps):
        action                           = agent.select_action(obs)
        next_obs, reward, term, trunc, _ = env.step(action)
        done                             = term or trunc
        agent.buffer.push(obs, action, reward, next_obs, done)
        ep_reward += reward
        obs = next_obs

        if done:
            episode_rewards.append(ep_reward)
            episode_timesteps.append(step + 1)
            ep_reward = 0.0
            obs, _ = env.reset()
            agent.epsilon = max(agent.eps_end, agent.epsilon * agent.eps_decay)

            n = len(episode_rewards)
            if n % log_interval == 0:
                avg = np.mean(episode_rewards[-log_interval:])
                print(f'Step {step+1:>7,} | Episode {n:4d} | '
                      f'Avg({log_interval}): {avg:.1f} | eps: {agent.epsilon:.3f}')

            if solve_threshold and n >= 100:
                if np.mean(episode_rewards[-100:]) >= solve_threshold:
                    print(f'Solved at step {step+1:,}!')
                    break

        q_loss, m_loss = agent.update()
        q_losses.append(q_loss)
        model_losses.append(m_loss)

    env.close()
    return {
        'episode_rewards':   episode_rewards,
        'episode_timesteps': episode_timesteps,
        'q_losses':          q_losses,
        'model_losses':      model_losses,
        'agent':             agent,
    }

print('train_dyna defined')

## A.1 — Train Dyna on CartPole-v1

Train DynaAgent with `planning_steps=10`. With `planning_steps=0` the agent reduces to a standard DQN baseline.

In [ ]:
set_seed(42)

# Dyna: DQN + 10 simulated planning steps per real step
results_dyna = train_dyna(
    env_name        = 'CartPole-v1',
    total_timesteps = 80_000,
    planning_steps  = 10,
    seed            = 42,
    lr              = 1e-3,
    batch_size      = 64,
    solve_threshold = 475.0,
    log_interval    = 20,
)

plot_dyna_results(results_dyna, title='Dyna — CartPole-v1', experiments_dir=EXPERIMENTS_DIR)
print(f'Final avg reward (last 20 ep): {np.mean(results_dyna["episode_rewards"][-20:]):.1f}')

## MCTS Planner

**Monte Carlo Tree Search (UCT)** plans by building a search tree from the current state. Each simulation:

1. **Selection** — traverse from root by UCB1 score until a leaf node
2. **Expansion** — add all unvisited children
3. **Simulation** — random rollout from the expanded node
4. **Backpropagation** — update visit counts and value estimates up to root

The action with the highest visit count at the root is returned.

We use `copy.deepcopy(env)` to fork the environment for each simulation — the real environment is never modified.

In [ ]:
class MCTSNode:
    """A single node in the MCTS tree — provided, no changes needed."""
    __slots__ = ('parent', 'action', 'children', 'visit_count', 'value_sum')

    def __init__(self, parent=None, action=None):
        self.parent      = parent
        self.action      = action
        self.children    = {}   # action_int -> MCTSNode
        self.visit_count = 0
        self.value_sum   = 0.0

    @property
    def q_value(self):
        return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0

    def ucb(self, c=1.41):
        if self.visit_count == 0:
            return float('inf')
        return self.q_value + c * np.sqrt(np.log(self.parent.visit_count) / self.visit_count)

    def is_leaf(self):
        return len(self.children) == 0


class MCTSPlanner:
    """
    UCT planner using the true Gymnasium environment as a model.

    Uses copy.deepcopy(env) to fork the environment for each simulation.

    Args:
        action_dim    (int):   Number of discrete actions.
        n_simulations (int):   MCTS iterations per action selection.
        depth         (int):   Maximum rollout depth per simulation.
        gamma         (float): Discount factor for rollout returns.
        c             (float): UCB exploration constant.
    """

    def __init__(self, action_dim, n_simulations=50, depth=20, gamma=1.0, c=1.41):
        self.action_dim    = action_dim
        self.n_simulations = n_simulations
        self.depth         = depth
        self.gamma         = gamma
        self.c             = c

    def select_action(self, env):
        """
        Run MCTS from the current env state and return the best action.

        Args:
            env: current Gymnasium env (deepcopied internally — not modified).

        Returns:
            int: action with highest visit count at root.

        Steps (repeat n_simulations times):
        1. sim_env = copy.deepcopy(env); node = root; G = 0.0; disc = 1.0
        2. Selection: while not leaf and not done and depth < self.depth:
               pick child with max UCB, step sim_env, accumulate G
        3. Expansion: if not done, add all action children;
               pick a random child, step sim_env, accumulate G
        4. Simulation (random rollout): step sim_env with random actions
               until done or max depth, accumulate G
        5. Backpropagation: walk from node to root,
               n.visit_count += 1; n.value_sum += G
        Return: max(root.children, key=lambda a: root.children[a].visit_count)
        """
        # =====================================================================
        # TODO: Implement UCT search.
        #
        # Hints:
        #   root = MCTSNode()
        #   root.visit_count = 1   # avoids log(0) in UCB of direct children
        #   for _ in range(self.n_simulations):
        #       sim_env = copy.deepcopy(env)
        #       ... (see docstring above)
        # =====================================================================
        raise NotImplementedError('Implement MCTSPlanner.select_action()')

print('MCTSNode and MCTSPlanner defined')

In [ ]:
def evaluate_mcts(env_name='CartPole-v1', n_simulations=50, depth=20,
                  num_episodes=20, seed=0):
    """
    Evaluate an MCTSPlanner on a given environment.

    Args:
        env_name      (str): Gymnasium environment id.
        n_simulations (int): MCTS iterations per step.
        depth         (int): Rollout depth per simulation.
        num_episodes  (int): Number of evaluation episodes.
        seed          (int): Base random seed.

    Returns:
        list: Episode rewards.
    """
    env     = gym.make(env_name)
    planner = MCTSPlanner(env.action_space.n,
                          n_simulations=n_simulations, depth=depth)
    rewards = []

    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_reward = 0.0
        for _ in range(1000):
            action = planner.select_action(env)
            obs, r, term, trunc, _ = env.step(action)
            ep_reward += r
            if term or trunc:
                break
        rewards.append(ep_reward)
        print(f'Episode {ep+1:2d}/{num_episodes}: reward = {ep_reward:.1f}')

    env.close()
    print(f'\nMean ± Std: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}')
    return rewards

print('evaluate_mcts defined (SOLUTION)')

## A.2 — Evaluate MCTS on CartPole-v1

Run MCTS with `n_simulations=50` and `depth=20` for 20 episodes.
Note: MCTS does **not** learn — it plans from scratch at each step using the true env.

In [ ]:
mcts_rewards = evaluate_mcts(
    env_name      = 'CartPole-v1',
    n_simulations = 50,
    depth         = 20,
    num_episodes  = 20,
    seed          = 0,
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(mcts_rewards) + 1), mcts_rewards, color='steelblue', alpha=0.7)
ax.axhline(np.mean(mcts_rewards), color='red', linestyle='--',
           label=f'Mean = {np.mean(mcts_rewards):.1f}')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.set_title('MCTS (n=50, depth=20) on CartPole-v1')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---

# Part B — Experiments

---

**For all experiments:** Run at least **3 different random seeds** and report mean ± std.

## B.1 — Ablation: Planning Steps in Dyna

Sweep `planning_steps` ∈ {0, 5, 20, 100}.
At what point does more planning stop helping (or start hurting due to model error compounding)?

In [ ]:
seeds = [42, 123, 456]
k_values = [0, 5, 20, 100]

ablation_results = {}
for k in k_values:
    ablation_results[f'k = {k}'] = [
        train_dyna(env_name='CartPole-v1', total_timesteps=80_000,
                   planning_steps=k, seed=s, lr=1e-3, batch_size=64,
                   solve_threshold=None)
        for s in seeds
    ]

plot_comparison(ablation_results,
                title='Dyna — Planning Steps Ablation (CartPole-v1)',
                experiments_dir=EXPERIMENTS_DIR)

print('Mean ± Std (last 20 episodes):')
for name, runs in ablation_results.items():
    final = [np.mean(r['episode_rewards'][-20:]) for r in runs]
    print(f'  {name}: {np.mean(final):.1f} ± {np.std(final):.1f}')

## B.2 — MCTS: Effect of Simulation Count

Sweep `n_simulations` ∈ {5, 20, 50, 200}.
More simulations = better planning but higher compute cost per step.
What is the minimum number of simulations needed to solve CartPole reliably?

In [ ]:
sim_counts = [5, 20, 50, 200]
seeds_mcts = [0, 1, 2]

mcts_results = {}
for n in sim_counts:
    episode_lists = []
    for s in seeds_mcts:
        ep_rewards = evaluate_mcts(
            env_name='CartPole-v1',
            n_simulations=n, depth=20,
            num_episodes=10, seed=s * 100,
        )
        episode_lists.append(ep_rewards)
    mcts_results[f'n_sim = {n}'] = episode_lists

# Summary bar chart
labels = list(mcts_results.keys())
means  = [np.mean([np.mean(ep) for ep in runs]) for runs in mcts_results.values()]
stds   = [np.std( [np.mean(ep) for ep in runs]) for runs in mcts_results.values()]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color='steelblue', alpha=0.7)
ax.set_ylabel('Mean Episode Reward'); ax.set_xlabel('Simulations per Step')
ax.set_title('MCTS — Effect of Simulation Count on CartPole-v1')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
from utils import _save_plot
_save_plot(fig, 'MCTS Simulation Count', EXPERIMENTS_DIR)
plt.show()

print('Mean ± Std episode reward by simulation count:')
for k, (m, s) in zip(labels, zip(means, stds)):
    print(f'  {k}: {m:.1f} ± {s:.1f}')